# Campaña de técnicas — ALL_ROOTS

Vía recomendada: **ALL_ROOTS**. Añade el input V01 y activa Internet para el clone oficial. Default: sólo benchmark. Para campaña completa cambia únicamente RUN_MODE a FULL_CAMPAIGN y ejecuta Run all. RESUME permanece True.

196 dimensiones locales repartidas en quince estudios; 59 entradas semánticas y cuatro parámetros reactivos T0 fijos. Los resultados se devuelven para revisión humana, sin promoción CANON.


In [ ]:
RUN_MODE = "BENCHMARK_ONLY"
RESUME = True
ENTRYPOINT = 'ALL_ROOTS'
# Modes: BENCHMARK_ONLY, STRUCTURAL_SCREEN, NUMERIC_SEARCH, VALIDATE, FULL_CAMPAIGN


In [ ]:
import sys, hashlib, zipfile, json, os, subprocess, shutil, base64
from pathlib import Path
sys.dont_write_bytecode = True
INPUT_ROOT = Path("/kaggle/input")  # reserved for optional resume checkpoint inputs only
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = "https://github.com/Shein25/La-Grulla-Blanca.git"
ASSET_COMMIT = "ac184bf496277eda78493262a1255f245d5c1d97"
ASSET_BASE = "experimentos/balance_nuevo/techniques_kaggle_heavy_v0_4/assets"
EXPECTED_ZIP_SHA = "dd4032c12a574d071411a505565f970473baaa84c17d4302435a9a206938d63b"
EXPECTED_MANIFEST_SHA = "85465dd566690674b94eab91bdd0aebf1053aef76d8373335c5c94aada08bdf6"
PARTS = [f"TECHNIQUES_KAGGLE_HEAVY_INPUTS_V01_1_HOTFIX.part{i:03d}.b64" for i in range(1,6)]

def digest(raw): return hashlib.sha256(raw).hexdigest()
def run_git(*args, cwd=None):
    p = subprocess.run(["git", *args], cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.returncode:
        raise RuntimeError("GIT_ASSET_BOOTSTRAP_FAILED:\n" + p.stdout)
    return p.stdout.strip()

def authenticate(root):
    manifest = root / "MANIFEST_SHA256.txt"
    if digest(manifest.read_bytes()) != EXPECTED_MANIFEST_SHA:
        raise RuntimeError("INPUT_PACKAGE_AUTHORITY_MISMATCH")
    names=set()
    for line in manifest.read_text(encoding="utf-8").splitlines():
        expected,name=line.split("  ",1)
        if name in names or "/" in name or "\\" in name or ":" in name:
            raise RuntimeError("INVALID_INPUT_MANIFEST")
        names.add(name)
        if digest((root/name).read_bytes()) != expected:
            raise RuntimeError("INPUT_HASH_MISMATCH:"+name)
    actual={p.name for p in root.iterdir() if p.is_file()}-{"MANIFEST_SHA256.txt"}
    if actual != names:
        raise RuntimeError("INPUT_FILE_SET_MISMATCH")
    return root

ASSET_REPO = WORKING / "La-Grulla-Blanca-Kaggle-Assets"
if ASSET_REPO.exists():
    shutil.rmtree(ASSET_REPO)
run_git("clone","--filter=blob:none","--no-checkout",OFFICIAL_REPO,str(ASSET_REPO))
run_git("checkout","--detach",ASSET_COMMIT,cwd=ASSET_REPO)
actual_commit=run_git("rev-parse","HEAD",cwd=ASSET_REPO)
if actual_commit != ASSET_COMMIT:
    raise RuntimeError(f"ASSET_COMMIT_MISMATCH:{actual_commit}!={ASSET_COMMIT}")

asset_dir=ASSET_REPO/ASSET_BASE
encoded="".join((asset_dir/p).read_text(encoding="ascii") for p in PARTS)
zip_bytes=base64.b64decode(encoded,validate=True)
if digest(zip_bytes) != EXPECTED_ZIP_SHA:
    raise RuntimeError("GIT_ASSET_ZIP_SHA_MISMATCH")

selected=WORKING/"TECHNIQUES_KAGGLE_HEAVY_INPUTS_V01_1_HOTFIX.zip"
selected.write_bytes(zip_bytes)
INPUTS=WORKING/"TECHNIQUES_LAB_INPUTS_V01"
if INPUTS.exists():
    shutil.rmtree(INPUTS)
with zipfile.ZipFile(selected) as z:
    if z.testzip() is not None:
        raise RuntimeError("CORRUPTED_INPUT_ZIP")
    if any("/" in n or "\\" in n or ":" in n for n in z.namelist()):
        raise RuntimeError("INVALID_INPUT_ZIP_PATH")
    if sum(i.file_size for i in z.infolist()) > 32*1024*1024:
        raise RuntimeError("INPUT_ZIP_SIZE_LIMIT")
    INPUTS.mkdir()
    for name in z.namelist():
        (INPUTS/name).write_bytes(z.read(name))
authenticate(INPUTS)
print("Git asset commit authenticated:", ASSET_COMMIT)
print("Input package authenticated:", INPUTS)


In [ ]:
import subprocess, importlib.metadata
if sys.version_info < (3, 11): raise RuntimeError("PYTHON_3_11_OR_NEWER_REQUIRED")
DEPENDENCIES = json.loads((INPUTS / "CAMPAIGN_CONFIG.json").read_text(encoding="utf-8"))["DEPENDENCIES"]
DEPENDENCY_TARGET = WORKING / "TECHNIQUES_DEPENDENCIES_V01_1"
lock_file = DEPENDENCY_TARGET / "PINNED_DEPENDENCIES.json"
if not lock_file.exists() or json.loads(lock_file.read_text()) != DEPENDENCIES:
    import shutil
    if DEPENDENCY_TARGET.exists(): shutil.rmtree(DEPENDENCY_TARGET)
    DEPENDENCY_TARGET.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--quiet", "--ignore-installed", "--no-warn-script-location", "--target", str(DEPENDENCY_TARGET)] + [k + "==" + v for k, v in DEPENDENCIES.items()], check=True)
    lock_file.write_text(json.dumps(DEPENDENCIES), encoding="utf-8")
print("Pinned CPU dependencies ready in isolated subprocess environment")


In [ ]:
request = {"inputs": str(INPUTS), "working": str(WORKING), "input_root": str(INPUT_ROOT), "RUN_MODE": RUN_MODE, "resume": RESUME, "scope": ENTRYPOINT}
request_file = WORKING / "TECHNIQUES_RUN_REQUEST.json"
request_file.write_text(json.dumps(request), encoding="utf-8")
environment = dict(os.environ)
environment["PYTHONPATH"] = str(DEPENDENCY_TARGET) + os.pathsep + str(INPUTS)
environment["PYTHONDONTWRITEBYTECODE"] = "1"
command = [sys.executable, "-u", "-B", str(INPUTS / "campaign_entrypoint.py"), "--run", str(request_file)]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=environment) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
        if line.startswith("CHECKPOINT_ZIP_READY "):
            from IPython.display import display, FileLink
            checkpoint = Path(line.strip().split(" ", 1)[1])
            if checkpoint.is_file(): display(FileLink(os.path.relpath(checkpoint, Path.cwd())))
    status = process.wait()
if status: raise RuntimeError("CAMPAIGN_FAILED: inspect the exact evidence above; previous checkpoints are preserved")
from IPython.display import display, FileLink
artifacts = WORKING / "TECHNIQUES_CAMPAIGN_V01"
for filename in ["TECHNIQUES_HEAVY_CHECKPOINT.zip", "RESULTS_TECHNIQUES_ALL_ROOTS_HEAVY.zip"] + ["RESULTS_" + r + "_HEAVY.zip" for r in ["FUEGO", "METAL", "AGUA", "TIERRA", "VIENTO"]]:
    target = artifacts / filename
    if target.exists(): display(FileLink(os.path.relpath(target, Path.cwd())))
